# 동현 — 오늘 본선 · Qwen3-VL-8B · 640px · 전량 · 2 epoch

어제 리더보드 **0.9407** 을 낸 설정 그대로이고, **시드 1** 실행 (팀원들과 다른 시드) 입니다. 설정은 다 들어가 있으니 **그대로 모두 실행**하세요.

**예상**: 학습 약 4시간 + dev 채점 15분 + test 추론 45분 ≈ **5시간** (10시 반 시작이면 15시 반 종료)

768px 2ep 은 6시간 45분이라 오늘 시작 시각으로는 18시를 넘겨서, 어제 0.9407 을 낸 640px 설정으로 갑니다. 고해상도 학습 확인은 한희(768px)가 맡습니다.

---

> 환경은 어제 이미 준비됨 → 2번 설치 셀은 "이미 설치됨" 으로 넘어갑니다.

### 실행 순서
0. **다른 노트북 커널 전부 종료** → 터미널에서 `nvidia-smi` 로 Memory-Usage 가 1GB 안팎인지 확인
1. 커널을 **venv_unsloth** 로 선택
2. **모두 실행** → 2번 설치 셀이 설치 후 **일부러 멈춤** → 커널 재시작 → 다시 모두 실행
3. 모델 다운로드(약 7GB)는 처음 한 번만

### 중간에 눈으로 확인할 곳
| 셀 | 정상 |
|---|---|
| 5번 | `min/max_pixels` 출력 · 이미지 토큰 약 400개 |
| 7번 | `loss 대상 디코딩` 이 정답 한 글자 + `<|im_end|>` |
| 8번 | 제로샷 예측이 한 글자에 몰리지 않음 |
| 9번 | 파일럿이 "시간 안에 들어옵니다" 라고 하면 학습 시작 |

### 끝나면
- 캐글 제출 (11번 마지막 줄이 설명란 문구)
- `output/ckpt/<RUN_NAME>/` 폴더 통째로 공유 드라이브 업로드 — **앙상블 재료라 이게 제일 중요**
- 막히면 그 화면 캡처해서 동현에게

In [ ]:
CONFIG = {
    "OWNER": "동현",
    "MODEL": "qwen3-vl-8b-4bit",
    "IMAGE_SIZE": 640,
    "N_TRAIN": 6714,             # 전량
    "EPOCHS": 2,                  # 전량 2ep (어제 0.9407 설정)
    "EARLY_STOP": True,
    "LR": 2e-4,
    "LORA_R": 32,
    "LORA_ALPHA": 64,
    "FINETUNE_VISION": True,
    "AUG": False,
    "SEED": 1,
    "BATCH_SIZE": 1,              # 16GB 라 1, 대신 누적 8 로 실효 배치 8 (이전 레시피와 동일)
    "GRAD_ACCUM": 8,
    "N_DEV_EPOCH": 1000,          # epoch 마다 채점할 dev 문항 수 (고정 부분집합). 시간 절약용
    "RUN_TEST": True,
    "END_BY": "17:40",            # 이 시각까지 test 추론까지 끝내는 걸 목표로 계산
    "MAX_HOURS": 6.5,
    "PILOT": True,
    # ---------------- 바꾸면 안 되는 것 ----------------
    "SPLIT_SEED": 42,
    "N_DEV": None,
    "MAX_SEQ_LEN": 2048,
}
PRESETS = {
    "qwen3-vl-8b-4bit": {"repo": "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit", "load_in_4bit": True,  "thinking": False},
    "qwen3.5-4b":       {"repo": "unsloth/Qwen3.5-4B",                            "load_in_4bit": False, "thinking": True},
    "qwen3.5-9b-4bit":  {"repo": "unsloth/Qwen3.5-9B",                            "load_in_4bit": True,  "thinking": True},
}
P = PRESETS[CONFIG["MODEL"]]
CONFIG["RUN_NAME"] = (f'{CONFIG["OWNER"]}_{CONFIG["MODEL"]}_px{CONFIG["IMAGE_SIZE"]}_n{CONFIG["N_TRAIN"]}'
                      f'_r{CONFIG["LORA_R"]}a{CONFIG["LORA_ALPHA"]}_lr{CONFIG["LR"]:g}_ep{CONFIG["EPOCHS"]}'
                      f'{"_vis" if CONFIG["FINETUNE_VISION"] else ""}{"_aug" if CONFIG["AUG"] else ""}_s{CONFIG["SEED"]}')
print("RUN_NAME:", CONFIG["RUN_NAME"])

# 1. 경로

In [ ]:
import os, sys
from pathlib import Path
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"     # 로짓 채점에 필요 (unsloth import 전에 설정)

def find_project_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data").is_dir() and (cand / "downloads").is_dir():
            return cand
    raise FileNotFoundError(f"data/ 와 downloads/ 가 있는 폴더를 찾지 못했습니다. 현재 위치: {p}")

PROJECT_ROOT = find_project_root()
DATA_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
CKPT_DIR   = OUTPUT_DIR / "ckpt"
SUB_DIR    = OUTPUT_DIR / "submissions"
for d in (OUTPUT_DIR, CKPT_DIR, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("python      :", sys.executable)    # venv_unsloth 가상환경 경로여야 함

# 2. 설치 (처음 한 번만)

In [ ]:
import importlib.util
_spec = importlib.util.find_spec("unsloth")
if _spec is not None and _spec.origin in (None, "namespace"):
    raise RuntimeError("작업 폴더에 'unsloth' 라는 이름의 폴더가 있어서 패키지를 가립니다. "
                       "가상환경 폴더라면 삭제하고 'venv_unsloth' 같은 다른 이름으로 다시 만드세요.")
if _spec is None:
    %pip install unsloth unsloth_zoo
    %pip install --force-reinstall --no-deps torch torchvision --index-url https://download.pytorch.org/whl/cu128
    if importlib.util.find_spec("triton") is None:
        %pip install triton-windows
    %pip install -q pandas pillow tqdm numpy huggingface_hub
    raise RuntimeError("설치 완료. 커널 재시작 후 다시 '모두 실행' 하세요. (이 에러는 정상입니다)")
else:
    print("unsloth 이미 설치됨 -> 건너뜀")

# 3. unsloth 불러오기 · 환경 확인

In [ ]:
# unsloth 는 transformers / peft 보다 먼저 import 해야 최적화가 적용됨
from unsloth import FastVisionModel
import torch, transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
assert "+cu" in torch.__version__, "CPU 전용 torch 입니다. 2번 설치 셀의 torch 재설치 줄을 다시 실행하세요"
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(), f"| VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB")

# 4. 데이터 로드 (월요일과 같은 문제집 · 평가셋)

In [ ]:
import re, math, random, time, json
import numpy as np
import pandas as pd
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

SEED = CONFIG["SEED"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_df_full = pd.read_csv(DATA_DIR / "train.csv")
dev_df        = pd.read_csv(DATA_DIR / "dev.csv")
test_df       = pd.read_csv(DATA_DIR / "test.csv")
sample_sub    = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train.csv 컬럼:", list(train_df_full.columns))
print("sample_submission.csv 컬럼:", list(sample_sub.columns))

# --- 이미지 경로 기준 자동 탐지 : path 컬럼이 'train/x.jpg' 인지 'data/train/x.jpg' 인지 모르므로 실측 ---
def resolve_image_base(df):
    rel = str(df["path"].iloc[0]).replace("\\", "/")
    for base in [DATA_DIR, PROJECT_ROOT]:
        if (base / rel).exists():
            return base
    # path 가 파일명만 있는 경우 (train/ 폴더 직속)
    for sub in ["train", "dev", "test"]:
        if (DATA_DIR / sub / rel).exists():
            return DATA_DIR / sub
    raise FileNotFoundError(f"이미지를 찾지 못했습니다: path='{rel}' (DATA_DIR={DATA_DIR})")

IMG_BASE = resolve_image_base(train_df_full)
print("이미지 기준 경로:", IMG_BASE)

def image_path(rel):
    return IMG_BASE / str(rel).replace("\\", "/")

assert image_path(train_df_full["path"].iloc[0]).exists()
assert image_path(dev_df["path"].iloc[0]).exists()
assert image_path(test_df["path"].iloc[0]).exists()
print("train/dev/test 이미지 경로 확인 완료")

# --- 평가셋 결정 : dev.csv 에 정답 컬럼이 있으면 dev, 없으면 train 홀드아웃 ---
HOLDOUT_N = 300
perm = np.random.RandomState(CONFIG["SPLIT_SEED"]).permutation(len(train_df_full))

print("dev.csv 컬럼:", list(dev_df.columns))
from collections import Counter
_LET = ["a", "b", "c", "d"]

def to_letter(v, row):
    # 투표값을 a/b/c/d 로 정규화. 빈 투표(NaN)는 무시, 보기 텍스트로 적혀 있으면 a~d 컬럼과 대조
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    if s in _LET:
        return s
    for L in _LET:
        if str(row[L]).strip().lower() == s:
            return L
    return None

def majority(votes):
    # 최다 득표 보기. 1위 동률이면 None (평가에서 제외)
    top = Counter(v for v in votes if v).most_common()
    if not top or (len(top) > 1 and top[0][1] == top[1][1]):
        return None
    return top[0][0]

VOTE_COLS = sorted(c for c in dev_df.columns if re.fullmatch(r"answer\d+", c))
LABEL_CANDS = ["answer", "label", "gold", "target", "정답"]
dev_label = next((c for c in LABEL_CANDS if c in dev_df.columns), None)

if VOTE_COLS:
    # dev 는 문항당 여러 명이 답한 구조 -> 다수결을 정답으로, 1위 동률 문항은 제외
    pool_idx = perm                                   # 전량 학습 가능
    d = dev_df.copy()
    d["votes"]  = d.apply(lambda r: [to_letter(r[c], r) for c in VOTE_COLS], axis=1)
    d["answer"] = d["votes"].apply(majority)
    n_tie = int(d["answer"].isna().sum())
    d = d[d["answer"].notna()]
    n_eval = min(CONFIG["N_DEV"] or len(d), len(d))
    dev_eval_df = d.sample(n=n_eval, random_state=CONFIG["SPLIT_SEED"]).reset_index(drop=True)
    unanimous = dev_eval_df["votes"].apply(lambda v: len({x for x in v if x}) == 1).mean()
    print(f"투표 컬럼 {VOTE_COLS} / dev {len(dev_df)}행 중 동률 {n_tie}행 제외")
    print(f"평가셋 만장일치 비율: {unanimous:.3f}")
    EVAL_SOURCE = f"dev 다수결({len(VOTE_COLS)}표, 동률제외)"
elif dev_label is not None:
    pool_idx = perm                                   # 전량 학습 가능
    dev_eval_df = dev_df.rename(columns={dev_label: "answer"})
    if CONFIG["N_DEV"] and CONFIG["N_DEV"] < len(dev_eval_df):
        dev_eval_df = dev_eval_df.sample(n=CONFIG["N_DEV"], random_state=CONFIG["SPLIT_SEED"])
    dev_eval_df = dev_eval_df.reset_index(drop=True)
    EVAL_SOURCE = f"dev.csv ('{dev_label}')"
else:
    # 순열 마지막 300개 = 어떤 N_TRAIN 으로 학습해도 절대 포함되지 않는 행
    pool_idx = perm[:-HOLDOUT_N]
    dev_eval_df = train_df_full.iloc[perm[-HOLDOUT_N:]].reset_index(drop=True)
    EVAL_SOURCE = f"train 홀드아웃 {HOLDOUT_N}"

# --- 학습 데이터 : 순열 앞에서 N_TRAIN 개 (df.sample(n, random_state) 와 동일한 결과) ---
n = CONFIG["N_TRAIN"]
if n > len(pool_idx):
    print(f"[알림] N_TRAIN {n} > 학습 풀 {len(pool_idx)} -> {len(pool_idx)} 로 제한")
    n = len(pool_idx)
train_df = train_df_full.iloc[pool_idx[:n]].reset_index(drop=True)

overlap = set(dev_eval_df["path"]) & set(train_df["path"])
assert not overlap, f"평가셋이 학습셋과 {len(overlap)}개 겹칩니다"
print(f"평가셋: {EVAL_SOURCE} {len(dev_eval_df)}행 (학습셋과 겹침 0)")

print(f"train {len(train_df)}행 / dev(평가) {len(dev_eval_df)}행 / test {len(test_df)}행")
print("정답 분포(train):", train_df["answer"].astype(str).str.strip().str.lower().value_counts().to_dict())

# 5. 모델 · LoRA · 해상도 · 이미지 증강

In [ ]:
PIX = CONFIG["IMAGE_SIZE"] ** 2
local = PROJECT_ROOT / "downloads" / "models" / P["repo"].split("/")[-1]
model_src = str(local) if (local / "config.json").exists() else P["repo"]
print("모델 경로:", model_src)

model, processor = FastVisionModel.from_pretrained(
    model_src,
    load_in_4bit = P["load_in_4bit"],
    max_seq_length = CONFIG["MAX_SEQ_LEN"],
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = CONFIG["FINETUNE_VISION"],
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = CONFIG["LORA_R"],
    lora_alpha = CONFIG["LORA_ALPHA"],
    lora_dropout = 0,
    bias = "none",
    random_state = CONFIG["SEED"],
)
# --- 해상도 고정 1 : processor 속성은 읽기 전용이라, 픽셀 한도를 넘겨 processor 를 다시 만든다 ---
from transformers import AutoProcessor
try:
    processor = AutoProcessor.from_pretrained(model_src, min_pixels=PIX, max_pixels=PIX)
except TypeError:
    processor = AutoProcessor.from_pretrained(model_src)   # 인자를 안 받는 버전 : 아래 load_image 의 사전 축소로 해상도 유지
tok = processor.tokenizer
tok.padding_side = "right"
ip = processor.image_processor
print("min/max_pixels:", getattr(ip, "min_pixels", None), getattr(ip, "max_pixels", None))

# --- 해상도 고정 2 : 이미지를 미리 목표 픽셀로 축소 (큰 사진은 빠른 축소 디코딩) ---
def load_image(rel):
    im = Image.open(image_path(rel))
    try:
        im.draft("RGB", (CONFIG["IMAGE_SIZE"] * 2, CONFIG["IMAGE_SIZE"] * 2))
    except Exception:
        pass
    im = im.convert("RGB")
    w, h = im.size
    s = (PIX / (w * h)) ** 0.5
    nw, nh = max(32, round(w * s / 32) * 32), max(32, round(h * s / 32) * 32)
    return im.resize((nw, nh), Image.BICUBIC)

# --- 이미지 증강 : 학습에만 사용. 글자를 읽는 과제라 뒤집기 · 큰 회전 · 큰 잘라내기 · 강한 블러는 넣지 않음 ---
import random
from PIL import ImageEnhance, ImageFilter
_aug_rng = random.Random(CONFIG["SEED"])

def augment(img):
    w, h = img.size
    if _aug_rng.random() < 0.5:                        # 가장자리 최대 5% 잘라낸 뒤 원래 크기로
        dx, dy = int(w * _aug_rng.uniform(0, 0.05)), int(h * _aug_rng.uniform(0, 0.05))
        l, t = _aug_rng.randint(0, dx), _aug_rng.randint(0, dy)
        img = img.crop((l, t, w - (dx - l), h - (dy - t))).resize((w, h), Image.BICUBIC)
    if _aug_rng.random() < 0.5:                        # ±3도 회전 (크기 유지)
        img = img.rotate(_aug_rng.uniform(-3, 3), resample=Image.BICUBIC, fillcolor=(128, 128, 128))
    if _aug_rng.random() < 0.8:                        # 밝기 · 대비 · 채도
        img = ImageEnhance.Brightness(img).enhance(_aug_rng.uniform(0.8, 1.2))
        img = ImageEnhance.Contrast(img).enhance(_aug_rng.uniform(0.8, 1.2))
        img = ImageEnhance.Color(img).enhance(_aug_rng.uniform(0.8, 1.2))
    if _aug_rng.random() < 0.1:                        # 가끔 아주 약한 블러
        img = img.filter(ImageFilter.GaussianBlur(radius=0.5))
    return img

_a = augment(load_image(train_df["path"].iloc[0]))
print("증강 확인: 크기", _a.size, "(원본과 같아야 이미지 토큰 수가 유지됨)")

_img = load_image(train_df["path"].iloc[0])
_thw = ip(images=[_img], return_tensors="pt")["image_grid_thw"][0]
n_img_tok = int(_thw.prod()) // (getattr(ip, "merge_size", 2) ** 2)
print(f"이미지 {_img.size} -> 토큰 {n_img_tok}개 (목표 약 {PIX // 1024})")
print(f"VRAM 사용: {torch.cuda.memory_allocated() / 1e9:.1f}GB")

# 6. 프롬프트 · 템플릿 (팀 공통)

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\\n"
        f"(a) {a}\\n(b) {b}\\n(c) {c}\\n(d) {d}\\n\\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def make_messages(row, img, with_answer):
    msgs = [
        {"role": "system", "content": SYSTEM_INSTRUCT},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])},
        ]},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": str(row["answer"]).strip().lower()})
    return msgs

def render(messages, gen):
    # Qwen3.5 는 thinking 모드가 기본 -> 꺼야 첫 토큰이 정답 글자가 됨
    kw = {"enable_thinking": False} if P["thinking"] else {}
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=gen, **kw)

LETTERS = ["a", "b", "c", "d"]
LETTER_IDS = []
for ch in LETTERS:
    ids = tok.encode(ch, add_special_tokens=False)
    assert len(ids) == 1, f"'{ch}' 가 단일 토큰이 아닙니다: {ids}"
    LETTER_IDS.append(ids[0])
print("letter ids:", dict(zip(LETTERS, LETTER_IDS)))

_r = train_df.iloc[0]
print("생성 프롬프트 끝부분:", repr(render(make_messages(_r, _img, False), True)[-70:]))

# 7. 데이터셋 · 라벨 마스킹 검증 (학습 이미지만 증강)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any

class VQAMCDataset(Dataset):
    def __init__(self, df, train=True):
        self.df, self.train = df.reset_index(drop=True), train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = load_image(row["path"])
        if self.train and CONFIG["AUG"]:
            img = augment(img)
        return {"messages": make_messages(row, img, self.train), "image": img}

@dataclass
class DataCollator:
    train: bool = True
    def __call__(self, batch):
        images = [s["image"] for s in batch]
        full   = [render(s["messages"], gen=not self.train) for s in batch]
        enc = processor(text=full, images=images, padding=True, return_tensors="pt")
        if self.train:
            prompts = [render(s["messages"][:-1], gen=True) for s in batch]
            p_enc = processor(text=prompts, images=images, padding=True, return_tensors="pt")
            prompt_lens = p_enc["attention_mask"].sum(dim=1)
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100
            for i, L in enumerate(prompt_lens):
                labels[i, :int(L)] = -100
            enc["labels"] = labels
        return enc

# --- 마스킹 검증 ---
_b = DataCollator(True)([VQAMCDataset(train_df.head(1))[0]])
_kept = _b["labels"][0][_b["labels"][0] != -100]
print("전체 토큰:", _b["labels"].shape[1], "/ loss 대상 토큰:", len(_kept))
print("loss 대상 디코딩:", repr(tok.decode(_kept)))
assert 1 <= len(_kept) <= 6, "loss 대상이 비었거나 너무 많습니다 -> 프롬프트/정답 경계가 어긋남 (캡처해서 동현에게)"

train_loader = DataLoader(VQAMCDataset(train_df, True), batch_size=CONFIG["BATCH_SIZE"],
                          shuffle=True, collate_fn=DataCollator(True), num_workers=0)
print("train batches:", len(train_loader))

# 8. 채점 함수 · 제로샷 확인

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

@torch.no_grad()
def predict_letters(df, desc="predict"):
    FastVisionModel.for_inference(model)
    preds, probs = [], []
    for i in tqdm(range(len(df)), desc=desc, unit="it"):
        row = df.iloc[i]
        img = load_image(row["path"])
        text = render(make_messages(row, img, False), gen=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**inputs).logits[:, -1, :].float()
        sel = logits[0, LETTER_IDS]
        preds.append(LETTERS[int(sel.argmax())])
        probs.append(torch.softmax(sel, dim=-1).cpu().tolist())
    return preds, probs

def accuracy(df, preds):
    gold = df["answer"].astype(str).str.strip().str.lower().tolist()
    return sum(p == g for p, g in zip(preds, gold)) / len(gold)

# --- 제로샷 확인 : 학습 전 모델이 정상적으로 답을 고르는지 (40문항) ---
_zp, _ = predict_letters(dev_eval_df.head(40), desc="제로샷 확인")
print(f"제로샷 40문항 정확도(dev 기준): {accuracy(dev_eval_df.head(40), _zp):.2f} | 예측 분포 {dict(Counter(_zp))}")
assert max(Counter(_zp).values()) < 36, "예측이 한 글자에 몰립니다 -> 템플릿/로짓 위치 문제 (캡처해서 동현에게)"

# 9. 파일럿 — 18시 안에 끝나는지 계산

In [ ]:
from datetime import datetime
dev_epoch_df = dev_eval_df.head(CONFIG["N_DEV_EPOCH"]).reset_index(drop=True)   # 고정 부분집합 (SPLIT_SEED 로 섞인 순서의 앞부분)

if CONFIG["PILOT"]:
    FastVisionModel.for_training(model)
    _opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG["LR"])
    t0, steps = time.time(), 20
    for step, batch in enumerate(tqdm(train_loader, total=steps, desc="pilot 학습"), start=1):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss
        loss.backward(); _opt.zero_grad(set_to_none=True)
        if step >= steps:
            break
    s_train = (time.time() - t0) / (steps * CONFIG["BATCH_SIZE"])
    del _opt; torch.cuda.empty_cache()
    t0 = time.time(); predict_letters(dev_eval_df.head(20), desc="pilot 채점"); s_inf = (time.time() - t0) / 20

    m_train = s_train * CONFIG["N_TRAIN"] * CONFIG["EPOCHS"] / 60
    m_dev   = s_inf * len(dev_epoch_df) * CONFIG["EPOCHS"] / 60
    m_test  = s_inf * len(test_df) / 60 if CONFIG["RUN_TEST"] else 0
    total   = m_train + m_dev + m_test
    end_by  = datetime.combine(datetime.now().date(), datetime.strptime(CONFIG["END_BY"], "%H:%M").time())
    left    = (end_by - datetime.now()).total_seconds() / 60
    print(f"학습 샘플당 {s_train:.2f}초 | 추론 문항당 {s_inf:.2f}초")
    print(f"예상: 학습 {m_train:.0f}분 + epoch별 dev {m_dev:.0f}분 + test {m_test:.0f}분 = {total:.0f}분")
    print(f"{CONFIG['END_BY']} 까지 남은 시간: {left:.0f}분")
    if total > left:
        fit_ep = max(1, int((left - m_test) // ((s_train * CONFIG['N_TRAIN'] + s_inf * len(dev_epoch_df)) / 60)))
        fit_n  = int(((left - m_test - m_dev) * 60 / s_train) / CONFIG["EPOCHS"])
        print(f"!! 시간 초과 예상. 선택지: EPOCHS={fit_ep} 로 줄이기, 또는 N_TRAIN≈{max(500, fit_n)} 로 줄이기")
        raise RuntimeError("CONFIG 를 조정하고 다시 '모두 실행' 하세요 (설치·다운로드는 다시 안 함)")
    print("-> 시간 안에 들어옵니다. 학습 시작")
else:
    print("PILOT=False -> 건너뜀")

# 10. 학습 — epoch 마다 저장 · 채점, 최고 epoch 자동 선택

In [ ]:
import math, gc
from transformers import get_linear_schedule_with_warmup

RUN_DIR = CKPT_DIR / CONFIG["RUN_NAME"]; RUN_DIR.mkdir(parents=True, exist_ok=True)
FastVisionModel.for_training(model)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=CONFIG["LR"])
total = CONFIG["EPOCHS"] * math.ceil(len(train_loader) / CONFIG["GRAD_ACCUM"])
scheduler = get_linear_schedule_with_warmup(optimizer, int(total * 0.03), total)

deadline, t_all = time.time() + CONFIG["MAX_HOURS"] * 3600, time.time()
history, best = [], {"epoch": 0, "acc": -1.0, "state": None}

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    FastVisionModel.for_training(model)
    t_ep, run_loss, seen = time.time(), 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['EPOCHS']}", unit="batch")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss / CONFIG["GRAD_ACCUM"]
        loss.backward()
        run_loss += loss.item() * CONFIG["GRAD_ACCUM"]; seen += 1
        if step % CONFIG["GRAD_ACCUM"] == 0:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); optimizer.zero_grad(set_to_none=True); scheduler.step()
            pbar.set_postfix({"loss": f"{run_loss / seen:.4f}"}); run_loss, seen = 0.0, 0
        if time.time() > deadline:
            break
    train_min = (time.time() - t_ep) / 60

    # --- epoch 끝 : 저장 + dev 채점 ---
    ep_dir = RUN_DIR / f"epoch{epoch}"
    model.save_pretrained(ep_dir); processor.save_pretrained(ep_dir)
    t_ev = time.time()
    preds, probs = predict_letters(dev_epoch_df, desc=f"dev epoch{epoch}")
    acc = accuracy(dev_epoch_df, preds)
    np.save(ep_dir / "dev_probs.npy", np.array(probs))
    history.append({"epoch": epoch, "dev_acc": round(acc, 4), "학습분": round(train_min, 1),
                    "채점분": round((time.time() - t_ev) / 60, 1)})
    print(f"[epoch {epoch}] dev {acc:.4f} | 학습 {train_min:.0f}분 | 누적 {(time.time() - t_all) / 60:.0f}분 | 저장 {ep_dir}")

    if acc > best["acc"]:
        best = {"epoch": epoch, "acc": acc,
                "state": {k: v.detach().to("cpu").clone() for k, v in model.state_dict().items() if "lora_" in k}}
    elif CONFIG["EARLY_STOP"]:
        print(f"-> dev 가 epoch {best['epoch']} ({best['acc']:.4f}) 보다 낮아서 학습을 멈춥니다")
        break
    if time.time() > deadline:
        print("-> 시간 상한 도달"); break

# --- 최고 epoch 모델로 되돌리기 ---
model.load_state_dict(best["state"], strict=False)
print(f"\\n선택: epoch {best['epoch']} (dev {best['acc']:.4f})")
print(pd.DataFrame(history).to_string(index=False))
del optimizer, scheduler
model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()

# 11. 기록 · test 추론 · 제출 파일

In [ ]:
acc = best["acc"]
row = {"RUN_NAME": CONFIG["RUN_NAME"], "OWNER": CONFIG["OWNER"], "MODEL": CONFIG["MODEL"],
       "IMAGE_SIZE": CONFIG["IMAGE_SIZE"], "N_TRAIN": len(train_df), "EPOCHS_max": CONFIG["EPOCHS"],
       "best_epoch": best["epoch"], "dev_acc": round(acc, 4), "n_dev": len(dev_epoch_df),
       "epoch별": " / ".join(f"ep{h['epoch']}={h['dev_acc']}" for h in history),
       "LR": CONFIG["LR"], "LORA_R": CONFIG["LORA_R"], "VIS": CONFIG["FINETUNE_VISION"], "AUG": CONFIG["AUG"], "quant": "4bit" if P["load_in_4bit"] else "bf16", "LB_public": None,
       "기록시각": time.strftime("%Y-%m-%d %H:%M")}
log_path = OUTPUT_DIR / "experiments_unsloth.csv"
log_df = pd.DataFrame([row])
if log_path.exists():
    log_df = pd.concat([pd.read_csv(log_path), log_df], ignore_index=True)
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

if CONFIG["RUN_TEST"]:
    t0 = time.time()
    test_preds, test_probs = predict_letters(test_df, desc="test")
    ID_COL, ANS_COL = sample_sub.columns[0], sample_sub.columns[1]
    submission = sample_sub.copy()
    submission[ANS_COL] = submission[ID_COL].map(dict(zip(test_df[ID_COL], test_preds)))
    assert submission[ANS_COL].isna().sum() == 0
    tag = f"{CONFIG['RUN_NAME']}_best-ep{best['epoch']}"
    sub_path = SUB_DIR / f"submission_{tag}.csv"
    submission.to_csv(sub_path, index=False)
    np.save(RUN_DIR / f"epoch{best['epoch']}" / "test_probs.npy", np.array(test_probs))
    print(f"저장: {sub_path} ({(time.time() - t0) / 60:.1f}분) | 분포 {submission[ANS_COL].value_counts().to_dict()}")
    print(f"캐글 설명란: {tag} | dev {acc:.4f}")

# 12. 끝나면
- [ ] 캐글 제출 — 설명란은 11번 마지막 줄 (`dev` 는 1,000문항 기준이라 월요일 2,216문항 점수와 직접 비교 금지, 리더보드로 판단)
- [ ] 선발대 결과(설치 · 마스킹 · 속도)를 팀에 공유
- [ ] `output/ckpt/<RUN_NAME>/` 공유 드라이브 업로드